In [1]:
import json
import textwrap
import copy
import logging
from tqdm import tqdm


logger = logging.getLogger(__name__)


In [2]:
import pickle
from pathlib import Path

test_set = pickle.load(open(Path.cwd().parent / "local_data" / "test_set.pkl", "rb"))
texts = [data["text"] for data in test_set]
pmids = [data["pmid"] for data in test_set]


Layer 1 and 2

In [3]:
INPUT = (texts, pmids)


In [4]:
from l1_decompose_expr import *

OUTPUT = {}
for i, (text, pmid) in tqdm(enumerate(zip(*INPUT)), total=len(INPUT[0])):
    expr_tree = expression_parse_recursive(
        text,
        parse_surface_epistemic_mode=parse_surface_epistemic_mode,
        parse_surface_logical_grouping=parse_surface_logical_grouping,
        parse_surface_logical_unary=parse_surface_logical_unary,
        epistemic_mode_schema=epistemic_mode_schema,
        logical_grouping_schema=logical_grouping_schema,
        logical_unary_schema=logical_unary_schema,
        max_depth=8
    )
    OUTPUT[i] = {
        "text": text,
        "pmid": pmid,
        "expr_tree": expr_tree,
    }


  8%|▊         | 8/100 [01:30<24:22, 15.90s/it]Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
 19%|█▉        | 19/100 [04:03<10:34,  7.84s/it]Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
 20%|██        | 20/100 [04:30<18:25, 13.82s/it]Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
 40%|████      | 40/100 [08:12<10:13, 10.23s/it]Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
 44%|████▍     | 44/100 [08:57<10:44, 11.52s/it]Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
 48%|████▊     | 48/100 [09:5

In [5]:
from l2_collect_valid_statements import *

for key, value in OUTPUT.items():
    valid_statements = valid_from_exp_tree(value['expr_tree'])
    OUTPUT[key]["valid_statements"] = valid_statements


In [6]:
print("OUTPUT:")
print(json.dumps(OUTPUT, indent=2))


OUTPUT:
{
  "0": {
    "text": "C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation.",
    "pmid": "25174320",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "ATOM": "C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation."
        }
      }
    },
    "valid_statements": [
      "C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation."
    ]
  },
  "1": {
    "text": "Furthermore, acetylation of p53 K120 by the MOF and MSL1v1 complex greatly enhanced the transcription activity of p53 (XREF_FIG).",
    "pmid": "19854137",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "ATOM": "acetylation of p53 K120 by the MOF and MSL1v1 complex greatly enhanced the transcription activity of p53"
        }
      }
    },
    "valid_statements": [
      "acetylation of p53 K120 by 

Layer 3 and 4

In [7]:
from l3_and_l4_one_pass import *


In [8]:
for key, item in tqdm(OUTPUT.items(), total=len(OUTPUT)):
    valid_statements = item['valid_statements']
    if len(valid_statements) == 0:
        continue
    else: 
        bio_rel_predicates = []
        bio_rels = []
        bio_rels_filtered = []
        for i, valid_statement in enumerate(valid_statements):
            try:
               val_statement_bio_predicates = extract_biomolecular_relationship_predicates(valid_statement)
               bio_rel_predicates.append(val_statement_bio_predicates)
               val_statement_bio_rels=extract_biomolecular_relationships(valid_statement, val_statement_bio_predicates)
               bio_rels.append(val_statement_bio_rels)
               val_statement_filtered_bio_rels = [rel for rel in val_statement_bio_rels if relationship_type_checker(str(rel))['valid'] == True]
               bio_rels_filtered.append(val_statement_filtered_bio_rels)
            except Exception as e:
                logger.warning(f"Error processing valid statement: {valid_statement}: {e}")
                bio_rel_predicates.append([])
                bio_rels.append([])
                bio_rels_filtered.append([])
        OUTPUT[key]["bio_rel_predicates"] = bio_rel_predicates
        OUTPUT[key]["bio_rels"] = bio_rels
        OUTPUT[key]["bio_rels_filtered"] = bio_rels_filtered


100%|██████████| 100/100 [20:43<00:00, 12.43s/it]


In [9]:
print("OUTPUT:")
print(json.dumps(OUTPUT, indent=2))


OUTPUT:
{
  "0": {
    "text": "C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation.",
    "pmid": "25174320",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "ATOM": "C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation."
        }
      }
    },
    "valid_statements": [
      "C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation."
    ],
    "bio_rel_predicates": [
      [
        "promote",
        "proliferate",
        "mediate",
        "acetylate"
      ]
    ],
    "bio_rels": [
      [
        {
          "predicate": "promote",
          "subject": "C5a",
          "object": "proliferation of human nasopharyngeal carcinoma cells"
        },
        {
          "predicate": "proliferate",
          "subject": "human nasopharyngeal carcinoma cells",
          "object": null
     

Layer 5

In [10]:
from l5_extract_indra_stmt_from_relationship import *
from indra_gpt.configs import BaseConfig, PostProcessorConfig
from indra_gpt.processors import PostProcessor

base_config = BaseConfig({"grounding": True, "grounding_strategy": "gilda", "num_samples": 100})
postprocessing_config = PostProcessorConfig(base_config)
post_processor = PostProcessor(postprocessing_config)


INFO: [2025-04-07 19:50:05] indra.preassembler.grounding_mapper.disambiguate - INDRA DB is not available for text content retrieval for grounding disambiguation.


In [11]:
all_preassembled_stmts = []
for key, item in tqdm(OUTPUT.items(), total=len(OUTPUT)):
    text = item['text']
    pmid = item['pmid']
    biomolecular_relationships = [rel for rels in item.get('bio_rels_filtered', []) for rel in rels]

    json_stmts = []
    for relationship in biomolecular_relationships:
        json_stmt = extract_indra_statement_from_relationship(str(relationship))
        json_stmts.append(json_stmt)
        # preprocessing block
        json_stmts = [post_processor.remove_empty_values(json_stmt) for json_stmt in json_stmts]
        json_stmts = [post_processor.update_evidence(stmt_json, text, pmid, postprocessing_config) 
                      for stmt_json in json_stmts] 
        json_stmts = [post_processor.add_evidence(stmt_json, str(relationship), pmid, postprocessing_config) 
                      for stmt_json in json_stmts] 
        for json_stmt in json_stmts:
            json_stmt.pop('id', None)
            json_stmt.pop('supports', None)
            json_stmt.pop('supported_by', None)
    OUTPUT[key]['json_stmts'] = json_stmts
    # Convert the json statements to raw INDRA statements.
    raw_stmts = post_processor.extract_stmts(json_stmts)
    OUTPUT[key]['raw_stmts'] = raw_stmts
    # Update the db_refs of the agents, because this is required for the grounding.
    for stmt in raw_stmts:
        try:
            for agent in stmt.agent_list():
                if agent and not agent.db_refs.get('TEXT', None):
                    agent.db_refs.update({'TEXT': agent.name})
        except Exception as e:
            logger.warning(f"Failed to update db_refs for agent: {stmt} — error: {e}")
    # Get preassembled statements.
    try:
        preassembled_stmts = post_processor.preassembly_pipeline(raw_stmts)
    except Exception as e: 
        logger.warning(e)
        preassembled_stmts = []
    OUTPUT[key]['preassembled_stmts'] = preassembled_stmts
    all_preassembled_stmts += preassembled_stmts


  0%|          | 0/100 [00:00<?, ?it/s]INFO: [2025-04-07 19:50:06] indra.pipeline.pipeline - Running the pipeline
INFO: [2025-04-07 19:50:06] indra.pipeline.pipeline - Calling filter_no_hypothesis
INFO: [2025-04-07 19:50:06] indra.tools.assemble_corpus - Filtering 0 statements to no hypothesis...
INFO: [2025-04-07 19:50:06] indra.tools.assemble_corpus - 0 statements after filter...
INFO: [2025-04-07 19:50:06] indra.pipeline.pipeline - Calling filter_no_negated
INFO: [2025-04-07 19:50:06] indra.tools.assemble_corpus - Filtering 0 statements to not negated...
INFO: [2025-04-07 19:50:06] indra.tools.assemble_corpus - 0 statements after filter...
INFO: [2025-04-07 19:50:06] indra.pipeline.pipeline - Calling custom_ground_statements
INFO: [2025-04-07 19:50:06] indra_gpt.processors.post_processors - Grounding 0 statements with GILDA...
INFO: [2025-04-07 19:50:06] indra_gpt.processors.post_processors - Grounded 0 statements with GILDA.
INFO: [2025-04-07 19:50:06] indra.pipeline.pipeline - Cal

In [12]:
OUTPUT


{0: {'text': 'C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation.',
  'pmid': '25174320',
  'expr_tree': {'DECLARATIVE': {'IDENTITY': {'ATOM': 'C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation.'}}},
  'valid_statements': ['C5a promotes the proliferation of human nasopharyngeal carcinoma cells through PCAF-mediated STAT3 acetylation.'],
  'bio_rel_predicates': [['promote', 'proliferate', 'mediate', 'acetylate']],
  'bio_rels': [[{'predicate': 'promote',
     'subject': 'C5a',
     'object': 'proliferation of human nasopharyngeal carcinoma cells'},
    {'predicate': 'proliferate',
     'subject': 'human nasopharyngeal carcinoma cells',
     'object': None},
    {'predicate': 'mediate', 'subject': 'PCAF', 'object': 'STAT3 acetylation'},
    {'predicate': 'acetylate', 'subject': 'STAT3', 'object': None}]],
  'bio_rels_filtered': [[]],
  'json_stmts': [],
  'raw_stmts': 

In [13]:
all_preassembled_stmts


[Acetylation(EP300(), TP73()),
 Activation(EGR1(), MMP14()),
 Activation(EGR1(), MMP2()),
 Activation(EGR1(), MMP9()),
 Activation(TCR(), CD28()),
 Activation(TGFB(), ERK()),
 Phosphorylation(TGFB(), SMAD4()),
 Activation(VEGF(), RAC1()),
 Activation(TGFB1(), ADAM17()),
 Activation(WNT1(), PPARG()),
 Activation(GAS6(), Mer receptor tyrosine kinase(), kinase),
 Activation(GAS6(), AXL(), kinase),
 Activation(RAPGEF1(), JNK(), kinase),
 Activation(GZMB(), CASP10()),
 Activation(GZMB(), CASP3()),
 Activation(GZMB(), CASP7()),
 Activation(GZMB(), CASP8()),
 Activation(GZMB(), CASP9()),
 Activation(HGF(), DGKA()),
 Activation(HIF1A(), TP53()),
 Activation(IFNB(), PI3K()),
 Activation(IFNB(), AKT1()),
 Activation(IFNB(), IRF1(), transcription),
 Activation(IL18(), RAS()),
 Activation(IL18(), SRC(kinase), kinase),
 Activation(PRKCE(), IKBKB(), kinase),
 Activation(PTGER3(), G_i()),
 Activation(CDC42(), PIKFYVE(), kinase),
 Activation(ARF6(), PIKFYVE(), kinase),
 Activation(RHOA(), PIKFYVE(), k

In [ ]:
import pickle
import time
import os

today = time.strftime("%Y-%m-%d-%H")
output_path = Path.cwd().parent / "output" / "semantic_decomposition" / today / "preassembled_statements.pkl"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pickle.dump(all_preassembled_stmts, open(output_path, "wb"))
